# JAX Spectra Dataloader Example

This notebook shows how to use `scripts/jax_spectra_dataloader.py` for ML training:
- open a spectra Zarr store
- create train/val/test JAX dataloaders
- inspect batches
- run a minimal JAX training loop


In [1]:
import os
from pathlib import Path

# Force CPU backend by default so the notebook is stable on machines
# with incomplete accelerator plugin setups.
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import numpy as np
import zarr
import jax
import jax.numpy as jnp
import sys

# In notebooks, __file__ is not defined. Use current working directory as base.
scripts_path = str((Path().resolve() / "scripts"))
sys.path.insert(0, scripts_path)

from scripts.jax_spectra_dataloader import create_jax_spectra_dataloaders

In [3]:
candidate_paths = [
    Path("spectra_tiny.zarr"),
]

zarr_path = None
for path in candidate_paths:
    if path.exists():
        zarr_path = path
        break

if zarr_path is None:
    raise FileNotFoundError(
        "Could not find a spectra Zarr store. Update candidate_paths with your dataset path."
    )

print(f"Using dataset: {zarr_path}")


Using dataset: spectra_tiny.zarr


In [4]:
root = zarr.open_group(str(zarr_path), mode="r")
array_names = sorted(root.array_keys())
print("Arrays:", array_names)

n_rows = None
for name in ("flux", "params", "global_index", "model_id"):
    if name in root:
        n_rows = int(root[name].shape[0])
        break
print("Rows:", n_rows)

if "wavelength" in root:
    print("Wavelength points:", int(root["wavelength"].shape[0]))

if "param_names" in root:
    param_names = [str(x) for x in np.asarray(root["param_names"][:]).tolist()]
    print("param_names:", param_names)


Arrays: ['flux', 'global_index', 'model_id', 'mu_selected', 'mu_selected_index', 'param_names', 'params', 'physics_hash', 'schema_version', 'wavelength']
Rows: 100
Wavelength points: 400001
param_names: ['teff', 'logg', 'feh', 'vmicro', 'a', 'c', 'n', 'o', 'r', 's']


In [5]:
has_params = "params" in root and "param_names" in root
has_flux = "flux" in root

if has_params and has_flux:
    input_key = "params"
    target_key = "flux"
elif has_flux:
    input_key = "flux"
    target_key = None
else:
    raise ValueError("Need at least a 'flux' array, or both 'params' and 'flux'.")

input_features = None
if input_key == "params":
    param_names = [str(x) for x in np.asarray(root["param_names"][:]).tolist()]
    preferred = [name for name in ("teff", "logg", "feh", "vmicro") if name in param_names]
    input_features = preferred if preferred else param_names

print("input_key:", input_key)
print("target_key:", target_key)
print("input_features:", input_features)


input_key: params
target_key: flux
input_features: ['teff', 'logg', 'feh', 'vmicro']


In [6]:
loaders = create_jax_spectra_dataloaders(
    zarr_path=str(zarr_path),
    batch_size=32,
    input_key=input_key,
    target_key=target_key,
    input_features=input_features,
    train_fraction=0.8,
    val_fraction=0.1,
    normalize_inputs=(input_key == "params"),
    normalize_targets=False,
    seed=7,
)

for split in ("train", "val", "test"):
    loader = loaders[split]
    print(f"{split}: rows={loader.indices.size}, batches={len(loader)}")


train: rows=80, batches=3
val: rows=10, batches=1
test: rows=10, batches=1


In [7]:
batch = next(iter(loaders["train"]))
print("Batch keys:", list(batch.keys()))
print("inputs shape:", tuple(batch["inputs"].shape))
if "targets" in batch:
    print("targets shape:", tuple(batch["targets"].shape))
print("indices shape:", tuple(batch["indices"].shape))
print("JAX backend:", jax.default_backend())


Batch keys: ['inputs', 'indices', 'targets']
inputs shape: (32, 4)
targets shape: (32, 400001)
indices shape: (32,)
JAX backend: cpu


## Minimal training step

The next cell runs a tiny linear model update for a few batches.
If `targets` are present, it trains on `targets` (optionally sliced for speed).
If no `targets` are present, it falls back to predicting the per-spectrum mean flux.


In [8]:
def batch_to_xy(batch, max_target_dim=2048):
    x = batch["inputs"]
    if x.ndim == 1:
        x = x[:, None]

    if "targets" in batch:
        y = batch["targets"]
        if y.ndim == 1:
            y = y[:, None]
        if y.shape[1] > max_target_dim:
            y = y[:, :max_target_dim]
    else:
        # Unsupervised fallback: predict mean flux from each input vector.
        y = jnp.mean(x, axis=1, keepdims=True)

    return x, y

first_batch = next(iter(loaders["train"]))
x0, y0 = batch_to_xy(first_batch)

key = jax.random.PRNGKey(0)
params = {
    "W": 0.01 * jax.random.normal(key, (x0.shape[1], y0.shape[1]), dtype=x0.dtype),
    "b": jnp.zeros((y0.shape[1],), dtype=x0.dtype),
}

def predict(params, x):
    return x @ params["W"] + params["b"]

def loss_fn(params, x, y):
    pred = predict(params, x)
    return jnp.mean((pred - y) ** 2)

@jax.jit
def train_step(params, x, y, lr=1e-2):
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y)
    new_params = jax.tree_util.tree_map(lambda p, g: p - lr * g, params, grads)
    return new_params, loss

for step, batch in zip(range(5), loaders["train"]):
    x, y = batch_to_xy(batch)
    params, loss = train_step(params, x, y, lr=1e-2)
    print(f"step={step:02d} loss={float(loss):.6f} x={tuple(x.shape)} y={tuple(y.shape)}")


step=00 loss=1.000262 x=(32, 4) y=(32, 2048)
step=01 loss=1.000343 x=(32, 4) y=(32, 2048)
step=02 loss=1.000247 x=(16, 4) y=(16, 2048)
